# LAB 2 &ndash; Shuffling cards

In this second lab, you'll get acquainted with AES and learn how to cryptographically shuffle a deck of cards.

First thing to do: double-click on this text cell to write your <b style="color: red">HACKOOLIQUES</b>.

## 0) Bytes

Byte (« octets ») arrays in Python 3 can be thought of a list of integers between 0 and 255: 

In [13]:
a = bytes([72, 69, 76, 76, 79])

a

b'HELLO'

The corresponding ASCII characters are displayed for convenience (the `b` reminds us that it is not a regular string but a byte array), but the individual bytes should not be thought of as "characters" (integers are returned):

In [14]:
for c in a:
    print(c)

72
69
76
76
79


Also, be aware that most values don't correspond to printable ASCII characters and just won't display nicely:

In [15]:
b = bytes([145, 8, 203, 78, 23])
b

b'\x91\x08\xcbN\x17'

Hexadecimal values are displayed for bytes for which there is no better option; in general, this is the preferred way to look at a byte array (2 hexadecimal numbers for each byte).

In [16]:
b.hex()

'9108cb4e17'

It's easy to implement the XOR (or $\oplus$) operation for bytes since Python has a built-in bitwise-XOR operator for integers.

In [17]:
def xor(a: bytes, b: bytes):
    return bytes([x^y for x,y in zip(a,b)])

print("a      :", a.hex())
print("b      :", b.hex())
print("a xor b:", xor(a,b).hex())

a      : 48454c4c4f
b      : 9108cb4e17
a xor b: d94d870258


**To do:** Before you proceed, make sure everything above makes sense to you (experience shows that you will probably need to read it again in a couple of minutes)...

As a warm-up, make sure you are able to turn a string of hex digits (such as `'48454c4c4f'`) into a `bytes` object (such as `a`).

In [18]:
h = "0x48454c4c4f"
b = bytes.fromhex(h[2:])
b

b'HELLO'

## 1) Using AES

We will use the implementation of AES provided by the <a href="https://pypi.org/project/cryptography/">cryptography</a> library (`pip install cryptography` if needed).

In [26]:
import os

# Hazardous Materials: use in real-world applications only if you know what you're doing!
# cryptography also provides "idiot-proof" Recipes that should be preferred

from cryptography.hazmat.primitives.ciphers import Cipher
from cryptography.hazmat.primitives.ciphers.algorithms import AES
from cryptography.hazmat.primitives.ciphers.modes import ECB
from cryptography.hazmat.backends import default_backend

AES works with blocks of 128 bits, _i.e._ 16 bytes.

In [27]:
AES.block_size # in bits

128

So let us now initialize AES with a randomly generated 16-byte key. 

In [29]:
k = os.urandom(16)

print("Secret key:", k.hex())

cipher = Cipher(AES(k), ECB(), default_backend())

encryptor = cipher.encryptor()  # E(k, ) in the slides

Secret key: b038f9b0ce6d1fc4147acb0d989f726d


We can now start encrypting text. 

In [30]:
m = b"Don't forget to respect the block length. A reversible padding scheme is needed in general.#####"

encryptor = cipher.encryptor()
c = encryptor.update(m)

for i in range(len(c)//16):
    print(c[16*i:16*(i+1)].hex())

da25391f58a53cd138ac93eb76d97e48
0e7e70e0b67a7dfb6f5219d36bd3a7d6
cdcfc9c0eae5a9d9c0760a5a8eb99372
0b57da7d184cae90f873e710b1b1e1fb
a700a9b69fa5de6ba937d99af2db12a4
c79062fbefcc47df1cafba3fee622d78


In [31]:
decryptor = cipher.decryptor()
decryptor.update(c)

b"Don't forget to respect the block length. A reversible padding scheme is needed in general.#####"

Notice what happens when blocks repeat: 

In [32]:
m = b"Don't use ECB...Don't use ECB...Don't use ECB..."

c = encryptor.update(m)

for i in range(len(c)//16):
    print(c[16*i:16*(i+1)].hex())

962b844952af5f0e2c9989cae8cf7165
962b844952af5f0e2c9989cae8cf7165
962b844952af5f0e2c9989cae8cf7165


This breaks semantic security, since the attacker should not be allowed to know that the message has repeating blocks.

<b>To do</b>: Encrypt the same message using AES in cipher block chaining (CBC) mode "by hand" (<i>i.e.</i>, **don't** use the `cryptography` API except to encrypt single blocks, set your encryptor to ECB mode). Verify that Bob is able to decrypt the received ciphertext knowing only the shared secret key $k$.

In [57]:
k = os.urandom(16)
print(f"Alice transmet la clef k={k.hex()} par un canal sécurisé à Bob")


m = b"Don't use ECB...Don't use ECB...Don't use ECB..."

print(f'Alice encode le message "{m}" avec la clef k')
cipher_alice = Cipher(AES(k), ECB(), default_backend())
encryptor_alice = cipher_alice.encryptor()


blocks = [m[i:i+16] for i in range(0, len(m), 16)] # découpé en blocs de 16 octets
c = [encryptor_alice.update(b) for b in blocks]
c_hex = [c_i.hex() for c_i in c]
print(f'Alice transmets le chiffré suivant à Bob {c_hex}')
print(f"Eve intercepte le chiffré mais en l'absence de la clef elle ne peut que constater que le chiffré contient 3x la meme information")
print(f"Bloc1 == Bloc2, Bloc2 == Bloc3", c[0]==c[1], c[1]==c[2])
print(f'Bob en possession de la clef k {k.hex()} peut déchiffrer le message')
cipher_bob = Cipher(AES(k), ECB(), default_backend())
decryptor_bob = cipher_bob.decryptor()

d = [decryptor_bob.update(b) for b in c]
print(b"".join(d))



Alice transmet la clef k=220421ee6a182e6de9d7e404b9fd35d4 par un canal sécurisé à Bob
Alice encode le message "b"Don't use ECB...Don't use ECB...Don't use ECB..."" avec la clef k
Alice transmets le chiffré suivant à Bob ['04d11d55aecbde9263266154bbee2ae1', '04d11d55aecbde9263266154bbee2ae1', '04d11d55aecbde9263266154bbee2ae1']
Eve intercepte le chiffré mais en l'absence de la clef elle ne peut que constater que le chiffré contient 3x la meme information
Bloc1 == Bloc2, Bloc2 == Bloc3 True True
Bob en possession de la clef k 220421ee6a182e6de9d7e404b9fd35d4 peut déchiffrer le message
b"Don't use ECB...Don't use ECB...Don't use ECB..."


## 2) Shuffling cards, pt. 1

Let's turn to a seemingly unrelated problem: that of a casino wanting to "impredictably" (from the point of view of the players) but "reproducibly" (from its point of view) shuffle a deck of digital playing cards with a 128-bit shuffle key (probably generated from a master key using a CSPRNG). There are $52! \approx 2^{226}$ different ways to shuffle a deck of cards, so the whole description of a shuffle couldn't possibly fit in the shuffle key: it has to be securely derived from it.

The first, "easiest" way to do it would be to:

1. Encrypt all card values using the shuffle key.
2. Sort lexicographically the resulting ciphertexts.
3. Assign to every "plaincard" its position in the sorted "ciphercards" list.

Hence obtaining a shuffled list of the original cards.

<b>To do</b>: Do it! What is the first poker hand (first 5 cards) dealt from your shuffled card deck?

# CBC
c0 = random Initial Value
ci = E (k , m[i] ⊕ c[i −1])

In [34]:
from cryptography.hazmat.primitives.ciphers import Cipher
from cryptography.hazmat.primitives.ciphers.algorithms import AES
from cryptography.hazmat.primitives.ciphers.modes import ECB
from cryptography.hazmat.backends import default_backend
from os import urandom


ranks = ["2","3","4","5","6","7","8","9","10","J","Q","K","A"]
suits = ["♣","♦","♥","♠"]
deck = [r+s for s in suits for r in ranks]


k = urandom(16)      

cipher = Cipher(AES(k), ECB(), default_backend())
encryptor = cipher.encryptor() 

encrypted_deck = [ encryptor.update( card.encode('utf-8').ljust(16, b"\0") ) for card in deck ]
sorted_encrypted = sorted( encrypted_deck )
decryptor = cipher.decryptor()
first_hand = [ decryptor.update( crypted ).rstrip(b"\0").decode('utf-8') for crypted in sorted_encrypted[:5] ]
first_hand

['K♠', '5♦', '4♣', '3♥', '3♦']

## 3) Shuffling cards, pt. 2

The above method works well, but rapidly becomes incovenient when the number of objects to permute gets large. Suppose for example that we want to shuffle not 52, but 52000 cards: with the above method, getting the top 5 would require 52000 single AES evaluations. A more efficient solution (look up  "Format-preserving encryption" for more information) is the following:

<ol>
    <li>Write every card value as a 2-byte word.</li>
    <li>Given a card value $m$, apply to it 3 rounds of a Feistel network with inner function $f$, where        
        $ f(b) = $ first byte of the AES encryption of byte $b$ padded with 15 # signs.</li> 
    <li>Repeat step 2. until the resulting value falls in the $[0,51999]$ range; this is the value of the card at position $m$ after the permutation.
</ol>

<b>To do:</b> What are the 5 first cards we get out of this 52000-card deck? How many AES evaluations did this take?

In [40]:
from cryptography.hazmat.primitives.ciphers import Cipher
from cryptography.hazmat.primitives.ciphers.algorithms import AES
from cryptography.hazmat.primitives.ciphers.modes import ECB
from cryptography.hazmat.backends import default_backend
from os import urandom

k = urandom(16)      
cipher = Cipher(AES(k), ECB(), default_backend())
encryptor = cipher.encryptor() 

AES_CPT=0
def aes_encrypt(block):
    global AES_CPT
    AES_CPT+=1
    return encryptor.update(block)

def feistel(word, upperbound=52000, loops=3):
    def f(b):
        block = bytes([b])+b"#"*15
        encrypted = aes_encrypt(block)
        return encrypted[0]
    
    for _ in range(loops):
        left = word[0]
        right = word[1] 

        word =  bytes([right, left ^ f(right)])
        
    if (word[0] << 8) | word[1] >= upperbound:
        return feistel(word, upperbound=upperbound, loops=loops)
    return word

word = bytes([42, 43])
encrypted = feistel(word,52000, 3)
print(encrypted, AES_CPT)



b'_\x8c' 3
